# 02 - EEGNet

**Amac:** Kompakt, EEG'ye ozel bir CNN'in zaman/uzay ozelliklerini uctan uca ogrenip ogrenemeyecegini olcmek (Braindecode EEGNetv4).

**Protokol:** Adaylar tuning_seed=42 ile yalnizca validation'da degerlendirilir; denek basina bir aday secilir; secilen aday tohum [42,43,44] ile egitilir; her tohumda test'te bir kez degerlendirilir. Tohumlar bagimsiz katilimci gibi sayilmaz.

**Beklenen ciktilar:** `results/eegnet/` altinda metrikler, egitim ozeti, tahminler, kontrol noktalari (tohum-duyarli), gecmisler, toplu metrikler, sekiller.

In [ ]:
# --- Ortak baslangic: proje koku kesfi ve ice aktarmalar ---
import sys, json, warnings
from pathlib import Path

def _find_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for d in (p, *p.parents):
        if (d / "pyproject.toml").exists() and (d / "src" / "cho2017_benchmark").exists():
            return d
    return p

ROOT = _find_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cho2017_benchmark import paths
from cho2017_benchmark.config import resolve_config, dump_resolved_config
from cho2017_benchmark.reproducibility import set_seed, save_environment

set_seed(42)
print("Proje koku:", paths.PROJECT_ROOT)

In [ ]:
QUICK_MODE = False  # Yalnizca hata ayiklama icindir; bilimsel sonuc uretmez.
cfg = resolve_config("eegnet", quick_mode=QUICK_MODE)
if cfg.quick_mode:
    print("UYARI: QUICK_MODE sonuclari nihai bilimsel sonuc olarak kullanilamaz.")
    print("QUICK_MODE denekleri:", cfg.active_subjects())
print("Aktif denek sayisi:", len(cfg.active_subjects()), "| Cihaz politikasi:", cfg.device)
print("Siniflar:", cfg.class_names, "| Ornekleme:", cfg.srate_in, "->", cfg.srate_out, "Hz")

In [ ]:
# Gerekli veri dosyalarini ve manifestolari dogrula
from cho2017_benchmark.data.inspect_mat import load_layout_resolution
resolution = load_layout_resolution()
print("Layout dogrulandi mi:", resolution["validated"], "| yonelim:", resolution["orientation"])

manifest_path = paths.manifests_dir() / "split_manifest.csv"
assert manifest_path.exists(), f"Split manifestosu yok: {manifest_path}. Once create_splits.py calistirin."
split_manifest = pd.read_csv(manifest_path)
print("Split manifestosu:", split_manifest.shape, "| denek:", split_manifest.subject_id.nunique())

from cho2017_benchmark.data.preprocessing import PreprocessingConfig
from cho2017_benchmark.reproducibility import preprocessing_hash, split_manifest_hash
PP = PreprocessingConfig.from_config(cfg)
PREPROC_HASH = preprocessing_hash(PP.hashable())
SPLIT_HASH = split_manifest_hash(split_manifest)
print("preprocessing_hash:", PREPROC_HASH, "| split_manifest_hash:", SPLIT_HASH)

In [ ]:
import torch
from cho2017_benchmark.training.trainer import resolve_device
DEVICE = resolve_device(cfg.device)
print("Cihaz:", DEVICE, "| CUDA:", torch.cuda.is_available(), "| tohumlar:", cfg.seeds)
# Model girdi/cikti sekil dogrulamasi (egitimsiz)
from cho2017_benchmark.models.factory import build_model, assert_no_softmax, count_trainable_parameters
_m, _info = build_model(cfg.get("model.name"), cfg, n_chans=cfg.n_eeg, n_times=cfg.expected_n_times())
print("Model kaynak:", _info.get("source"), "| parametre:", count_trainable_parameters(_m))
print("Softmax kontrolu:", assert_no_softmax(_m, n_chans=cfg.n_eeg, n_times=cfg.expected_n_times()))
del _m

## Denek bazli egitim/degerlendirme dongusu
`run_subject_neural` aday secimi + cok-tohum + test degerlendirmesini yapar.

In [ ]:
from cho2017_benchmark.training.experiment import run_subject_neural
from cho2017_benchmark.data import prepare
from cho2017_benchmark.data.datasets import make_subject_loaders
from cho2017_benchmark.evaluation.metrics import compute_subject_metrics
import logging

MODEL = "eegnet"
base = paths.results_dir(MODEL)
for sub in ["tables", "predictions", "metrics", "figures", "checkpoints", "histories", "configs", "logs"]:
    (base / sub).mkdir(parents=True, exist_ok=True)
logging.basicConfig(filename=base / "logs" / "experiment.log", level=logging.INFO, force=True)
batch_size = int(cfg.get("training.batch_size", 32))

all_metrics, all_preds, all_summary, all_selected, all_cands, failures = [], [], [], [], [], []
for idx in cfg.active_subjects():
    sid = f"s{idx:02d}"
    try:
        rec = prepare.load_processed(idx)
        sd = make_subject_loaders(sid, rec, split_manifest, batch_size=batch_size, seed=cfg.seed)
        sm = split_manifest[split_manifest.subject_id == sid]
        split_method = sm["split_method"].iloc[0] if len(sm) else "unknown"
        out = run_subject_neural(MODEL, cfg, sd, split_method=split_method,
                                 split_manifest_hash=SPLIT_HASH, preprocessing_hash=PREPROC_HASH, device=DEVICE)
        all_metrics.extend(out["metrics"]); all_preds.extend(out["predictions"])
        all_summary.extend(out["training_summary"]); all_cands.extend(out["candidates"])
        all_selected.append({"subject_id": sid, "selected_candidate": out["selected_candidate"],
                              "model_source": out["model_source"],
                              "n_candidates_evaluated": len(cfg.get("candidates", []) or [1])})
        logging.info("%s ok", sid)
    except Exception as e:
        failures.append({"subject_id": sid, "error": str(e)})
        all_metrics.append(compute_subject_metrics(None, None, None, model_name=MODEL, subject_id=sid,
            split_method="", n_train=0, n_validation=0, n_test=0, seed=cfg.seed,
            split_manifest_hash=SPLIT_HASH, preprocessing_hash=PREPROC_HASH,
            status="failed", error_message=str(e)))
        logging.exception("%s basarisiz", sid)
print("Metrik satiri:", len(all_metrics), "| basarisiz denek:", len(failures))

## Ciktilari kaydet

In [ ]:
from cho2017_benchmark.reporting.result_writer import (
    write_subject_metrics, write_predictions, write_training_summary,
    write_selected_hyperparameters, write_aggregate_metrics)
from cho2017_benchmark.evaluation.statistics import aggregate_seeds_per_subject

write_subject_metrics(all_metrics, base / "tables" / "subject_metrics.csv")
write_predictions(all_preds, base / "predictions" / "test_predictions.parquet")
write_training_summary(all_summary, base / "tables" / "training_summary.csv")
write_selected_hyperparameters(pd.DataFrame(all_cands), base / "tables" / "selected_hyperparameters.csv")
metrics_df = pd.DataFrame(all_metrics)
per_subject = aggregate_seeds_per_subject(metrics_df)  # tohum-ortalamali, katilimci basina
write_aggregate_metrics(per_subject, base / "metrics" / "aggregate_metrics.json")
dump_resolved_config(cfg, base / "configs" / "resolved_config.yaml")
save_environment(base)
(base / "logs" / "failures.json").write_text(json.dumps(failures, indent=2), encoding="utf-8")
print("Katilimci basina ortalama dogruluk:",
      round(float(per_subject["accuracy"].mean()), 3) if len(per_subject) else float("nan"))

## Ozet sekiller (primary_seed = 42 ile havuzlanmis kafa karistirma matrisi)

In [ ]:
from cho2017_benchmark.evaluation import plots
ok = metrics_df[metrics_df["status"] == "ok"]
prim = ok[ok["seed"] == cfg.primary_seed]
if len(prim):
    plots.save_fig(plots.subject_accuracy_sorted(prim), base / "figures" / "subject_accuracy_sorted.png")
    plots.save_fig(plots.subject_accuracy_distribution(prim), base / "figures" / "subject_accuracy_distribution.png")
    cm = np.array([[int(prim["tn"].sum()), int(prim["fp"].sum())], [int(prim["fn"].sum()), int(prim["tp"].sum())]])
    plots.save_fig(plots.confusion_matrix_plot(cm, cfg.class_names, title=f"{MODEL} toplu (seed42)"),
                   base / "figures" / "aggregate_confusion_matrix.png")
# temsili egitim egrileri
import glob
hist_files = sorted(glob.glob(str(base / "histories" / "*.csv")))[:3]
if hist_files:
    fig, ax = plt.subplots(figsize=(8, 4))
    for hf in hist_files:
        h = pd.read_csv(hf); ax.plot(h["epoch"], h["val_balanced_accuracy"], label=Path(hf).stem)
    ax.set_xlabel("epoch"); ax.set_ylabel("val balanced accuracy"); ax.legend(fontsize=7)
    plots.save_fig(fig, base / "figures" / "representative_training_curves.png")
plt.show()

## Limitations
- Katilimci basina sinirli egitim verisi nedeniyle derin model asiri ogrenebilir.
- Tohumlar arasi degiskenlik raporlanir; tohumlar katilimci degildir.
- GPU/CPU cihaz farki gecikme karsilastirmasinda dikkate alinmalidir.

## Generated Files
`results/eegnet/tables/*.csv`, `predictions/test_predictions.parquet`, `checkpoints/<sid>/seed_<seed>/best.pt`, `histories/<sid>_seed_<seed>.csv`, `metrics/aggregate_metrics.json`, `figures/*.png`.

In [ ]:
expected = [base / "tables" / "subject_metrics.csv",
            base / "predictions" / "test_predictions.parquet",
            base / "tables" / "training_summary.csv",
            base / "metrics" / "aggregate_metrics.json"]
missing = [str(p) for p in expected if not p.exists()]
assert not missing, f"Beklenen ciktilar eksik: {missing}"
print("Tum beklenen ciktilar mevcut.")